# 

# *IMPORT LIB*


In [21]:
!pip install transformers
!pip install pytorch-crf
!pip install scikit-learn
!pip install tqdm
!pip install datasets transformers
from huggingface_hub import login

login(token="HUGGING FACE TOKEN")

In [2]:
import os
import random

import numpy as np

import pandas as pd

import torch

import torch.nn as nn

from tqdm import tqdm

from torch.utils.data import Dataset,DataLoader

from transformers import AutoTokenizer,AutoModel

from torchcrf import CRF

from sklearn.metrics import (

    accuracy_score,

    f1_score,

    precision_score,

    recall_score,

    confusion_matrix,

    classification_report

)

# **CONFIG**

In [37]:

import torch

CONFIG = {
    "dataset_name": "opennyaiorg/InRhetoricalRoles",
    "bert_model": "nlpaueb/legal-bert-base-uncased",
    "freeze_bert": False, 
    "num_labels": None,
    "max_length": 128,
    "embedding_dim": 768,
    "hidden_dim": 256,
    "batch_size": 1,
    "epochs": 10, 
    "learning_rate": 2e-5, 
    "paragraph_size": 4,
    "dropout": 0.2,
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}

# **UTILS**

In [4]:

import os
import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def save_model(model,path):
    torch.save(model.state_dict(),path)

def load_model(model,path,device):
    model.load_state_dict(torch.load(path,map_location=device))
    return model


def create_directory(path):
    os.makedirs(path,exist_ok=True)

# **PREPROCESSING**

In [5]:

from datasets import load_dataset
import re

def load_opennyai_data(split_name="train"):
    """
    Loads Hugging Face dataset and restructures it to match document processing pipeline.
    Output format: List of Dicts [{"sentences": [...], "labels": [...], "doc_id": "..."}]
    """
    print(f"Downloading and loading opennyaiorg/InRhetoricalRoles ({split_name})...")
    hf_dataset = load_dataset("opennyaiorg/InRhetoricalRoles", split=split_name)
    
    parsed_documents = []
    
    for idx, row in enumerate(hf_dataset):
        full_text = row["data"]["text"]
        annotations_list = row["annotations"]
        
        extracted_segments = []
        if annotations_list and len(annotations_list) > 0:
            results = annotations_list[0].get("result", [])
            for res in results:
                value = res.get("value", {})
                start_char = value.get("start")
                end_char = value.get("end")
                labels = value.get("labels", ["None"])
                label = labels[0] if labels else "None"
                
                segment_text = full_text[start_char:end_char].strip()
                if segment_text:
                    extracted_segments.append((segment_text, label))
                    
        doc_sentences = []
        doc_labels = []
        
        if extracted_segments:
            for seg_text, label in extracted_segments:
                sub_sentences = re.split(r'(?<=[.!?])\s+', seg_text)
                for sentence in sub_sentences:
                    sentence = sentence.strip()
                    if len(sentence) > 3: 
                        doc_sentences.append(sentence)
                        doc_labels.append(label)
        else:
            sub_sentences = re.split(r'(?<=[.!?])\s+', full_text)
            for sentence in sub_sentences:
                sentence = sentence.strip()
                if len(sentence) > 3:
                    doc_sentences.append(sentence)
                    doc_labels.append("None")
                    
        if doc_sentences:
            parsed_documents.append({
                "sentences": doc_sentences,
                "labels": doc_labels,
                "doc_id": f"opennyai_{split_name}_{idx}"
            })
            
    print(f"Successfully compiled {len(parsed_documents)} docs for {split_name} split.")
    return parsed_documents

# **DATASET**

In [6]:
import torch
from torch.utils.data import Dataset

class LegalSegDataset(Dataset):
    def __init__(self, documents):
        self.documents = documents

    def __len__(self):
        return len(self.documents)

    def __getitem__(self, idx):
        doc = self.documents[idx]
        return {
            "doc_id": doc["doc_id"],
            "sentences": doc["sentences"],
            "labels": doc["labels"]
        }

# **LABEL_MAPPING**

In [7]:
def create_label_mapping(

        train_docs

    ):

    all_labels=set()

    for doc in train_docs:

        for label in doc["labels"]:

            label=str(label).strip()

            if label=="":

                continue

            all_labels.add(label)


    label2id={

        label:i

        for i,label in enumerate(

            sorted(all_labels)

        )

    }


    id2label={

        i:label

        for label,i in label2id.items()

    }


    return label2id,id2label

# **BERT_LAYER**

In [8]:
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer

class LegalBERTLayer(nn.Module):
    def __init__(self, model_name, freeze=False):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name)
        self.freeze = freeze
        
        if freeze:
            for param in self.bert.parameters():
                param.requires_grad = False
        else:
            self.bert.gradient_checkpointing_enable()

    def forward(self, sentences, chunk_size=16):
        device = next(self.parameters()).device
        all_embeddings = []
        
        context = torch.no_grad() if self.freeze else torch.enable_grad()
        
        with context:
            for i in range(0, len(sentences), chunk_size):
                batch_sentences = sentences[i : i + chunk_size]
                
                encoding = self.tokenizer(
                    batch_sentences, 
                    padding=True, 
                    truncation=True, 
                    max_length=128, 
                    return_tensors="pt"
                )
                encoding = {k: v.to(device) for k, v in encoding.items()}
                
                output = self.bert(**encoding)
                cls_embeddings = output.last_hidden_state[:, 0, :]
                
                if self.freeze:
                    cls_embeddings = cls_embeddings.detach()
                    
                all_embeddings.append(cls_embeddings)
                
        return torch.cat(all_embeddings, dim=0)

# **BiLSTM_LAYER**

In [9]:

import torch.nn as nn

class BiLSTMLayer(nn.Module):

    def __init__(self,embedding_dim=768,hidden_dim=256):
        super().__init__()
        self.lstm=nn.LSTM(embedding_dim,hidden_dim,batch_first=True,bidirectional=True)

    def forward(self,x):
        output,_=self.lstm(x)
        return output

# **LAYER_SHIFT**

In [10]:

import torch
import torch.nn as nn

class LabelShiftLayer(nn.Module):


    def __init__(self,embedding_dim=768):
        super().__init__()
        self.fc=nn.Linear(embedding_dim,1)

    def forward(self,x):
        
        shift_logits=self.fc(x)
        return shift_logits.squeeze(-1)

# **CRF_LAYER**

In [11]:


import torch
import torch.nn as nn

from torchcrf import CRF


class CRFLayer(nn.Module):

    def __init__(self, num_labels):

        super().__init__()

        self.crf = CRF(

            num_tags=num_labels,

            batch_first=True

        )


    def forward(

            self,

            emissions,

            labels,

            mask

        ):

      

        mask = mask.bool()


        loss = -self.crf(

            emissions,

            labels,

            mask=mask,

            reduction="mean"

        )


        return loss


    def decode(

            self,

            emissions,

            mask

        ):

       

        mask = mask.bool()


        predictions = self.crf.decode(

            emissions,

            mask=mask

        )


        return predictions

# **BASELINE_MODEL**

In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.weight = weight 
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        if self.weight is not None:
            self.weight = self.weight.to(inputs.device)
            
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.weight)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        return focal_loss.sum()


class BaselineModel(nn.Module):
    def __init__(self, config, label2id):
        super().__init__()
        self.label2id = label2id
        self.legalbert = LegalBERTLayer(config["bert_model"], freeze=config["freeze_bert"])
        self.bilstm = BiLSTMLayer(embedding_dim=config["embedding_dim"], hidden_dim=config["hidden_dim"])
        self.label_shift = LabelShiftLayer(embedding_dim=512)
        self.classifier = nn.Linear(512, config["num_labels"])
        self.crf = CRFLayer(config["num_labels"])

        class_weights_tensor = torch.tensor(config["class_weights"], dtype=torch.float)
        self.focal_loss_fn = FocalLoss(weight=class_weights_tensor, gamma=2.5) 
        
        self.focal_loss_fn.to(config["device"])
        
    def forward(self, batch, device, max_doc_sentences=150):
        sentences = batch["sentences"]
        labels = batch["labels"]

        if len(sentences) > max_doc_sentences:
            sentences = sentences[:max_doc_sentences]
            labels = labels[:max_doc_sentences]

        label_ids = torch.tensor([self.label2id[label] for label in labels], dtype=torch.long, device=device)

        sentence_embeddings = self.legalbert(sentences)
        sentence_embeddings = sentence_embeddings.unsqueeze(0)
        bilstm_output = self.bilstm(sentence_embeddings).squeeze(0)

        emissions = self.classifier(bilstm_output)
        
        auxiliary_loss = self.focal_loss_fn(emissions, label_ids)

        emissions_crf = emissions.unsqueeze(0)
        labels_crf = label_ids.unsqueeze(0)
        mask = torch.ones(labels_crf.shape, dtype=torch.bool, device=device)

        crf_loss = self.crf(emissions_crf, labels_crf, mask)
        predictions = self.crf.decode(emissions_crf, mask)

        total_loss = crf_loss + auxiliary_loss
        return {"loss": total_loss, "predictions": predictions}

# **EVALUATE**

In [13]:


from sklearn.metrics import (

    accuracy_score,

    f1_score,

    precision_score,

    recall_score,

    confusion_matrix,

    classification_report

)


def evaluate(

        true_labels,

        pred_labels

    ):


    metrics = {}


    metrics["accuracy"] = accuracy_score(

        true_labels,

        pred_labels

    )


    metrics["macro_f1"] = f1_score(

        true_labels,

        pred_labels,

        average="macro",

        zero_division=0

    )


    metrics["weighted_f1"] = f1_score(

        true_labels,

        pred_labels,

        average="weighted",

        zero_division=0

    )


    metrics["precision"] = precision_score(

        true_labels,

        pred_labels,

        average="macro",

        zero_division=0

    )


    metrics["recall"] = recall_score(

        true_labels,

        pred_labels,

        average="macro",

        zero_division=0

    )


    metrics["confusion_matrix"] = confusion_matrix(

        true_labels,

        pred_labels

    )


    metrics["classification_report"] = classification_report(

        true_labels,

        pred_labels,

        zero_division=0

    )


    return metrics

# **TRAIN**

In [14]:


import torch
from tqdm.auto import tqdm

def train_epoch(model, train_loader, optimizer, device):
    model.train()
    total_loss = 0.0
    
    progress_bar = tqdm(train_loader, desc="  Training Batches", leave=False)
    
    for step, batch in enumerate(progress_bar):
        optimizer.zero_grad()
        
        outputs = model(batch, device)
        loss = outputs["loss"]
        
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix({"batch_loss": f"{loss.item():.4f}"})
        
    return total_loss / len(train_loader)


def validate_epoch(model, val_loader, device):
 
    model.eval()
    total_loss = 0.0
    
    with torch.no_grad():
        for batch in val_loader:
            outputs = model(batch, device)
            loss = outputs["loss"]
            total_loss += loss.item()
            
    return total_loss / len(val_loader)


def train(model, train_loader, val_loader, optimizer, config):
    
    device = config["device"]
    epochs = config["epochs"]
    best_val_loss = float("inf")
    history = {"train_loss": [], "val_loss": []}
    
    print(f"\nStarting Model Training Pipeline for {epochs} Epochs...")
    print("-" * 50)
    
    for epoch in range(1, epochs + 1):
        print(f"\nEpoch {epoch}/{epochs}")
        
        avg_train_loss = train_epoch(model, train_loader, optimizer, device)
        
        avg_val_loss = validate_epoch(model, val_loader, device)
        
        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)
        
        print(f"-> Epoch {epoch} Results | Train Avg Loss: {avg_train_loss:.4f} | Val Avg Loss: {avg_val_loss:.4f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), "best_model.pth")
            print("   ✨ New best validation checkpoint saved to 'best_model.pth'")
            
    return history

# **MAIN**

In [38]:


import os
import re
import torch
import numpy as np
from collections import Counter
from torch.utils.data import DataLoader

try:
    from datasets import load_dataset
except ImportError:
    print("Installing missing datasets library...")
    os.system("pip install datasets")
    from datasets import load_dataset


def identity_collate(batch):
    return batch[0]


def load_opennyai_data(split_name="train"):
    print(f"\n[Data] Fetching opennyaiorg/InRhetoricalRoles ({split_name} split)...")
    
    hf_split = "dev" if split_name == "val" else split_name
    hf_dataset = load_dataset("opennyaiorg/InRhetoricalRoles", split=hf_split)
    
    parsed_documents = []
    
    for idx, row in enumerate(hf_dataset):
        full_text = row["data"]["text"]
        annotations_list = row["annotations"]
        
        extracted_segments = []
        if annotations_list and len(annotations_list) > 0:
            results = annotations_list[0].get("result", [])
            for res in results:
                value = res.get("value", {})
                start_char = value.get("start")
                end_char = value.get("end")
                labels = value.get("labels", ["None"])
                label = labels[0] if labels else "None"
                
                segment_text = full_text[start_char:end_char].strip()
                if segment_text:
                    extracted_segments.append((segment_text, label))
                    
        doc_sentences = []
        doc_labels = []
        
        if extracted_segments:
            for seg_text, label in extracted_segments:
                sub_sentences = re.split(r'(?<=[.!?])\s+', seg_text)
                for sentence in sub_sentences:
                    sentence = sentence.strip()
                    if len(sentence) > 3:  
                        doc_sentences.append(sentence)
                        doc_labels.append(label)
        else:
            sub_sentences = re.split(r'(?<=[.!?])\s+', full_text)
            for sentence in sub_sentences:
                sentence = sentence.strip()
                if len(sentence) > 3:
                    doc_sentences.append(sentence)
                    doc_labels.append("None")
                    
        if doc_sentences:
            parsed_documents.append({
                "sentences": doc_sentences,
                "labels": doc_labels,
                "doc_id": f"opennyai_{split_name}_{idx}"
            })
            
    print(f" -> Compiled {len(parsed_documents)} complete document sequences.")
    return parsed_documents


print("Current workspace directory :", os.getcwd())
train_docs = load_opennyai_data(split_name="train")
val_docs = load_opennyai_data(split_name="dev")

label2id, id2label = create_label_mapping(train_docs)
CONFIG["num_labels"] = len(label2id)

counter = Counter()
for doc in train_docs:
    counter.update(doc["labels"])

total = sum(counter.values())
weights = []
for label in sorted(label2id.keys()):
    weights.append(
        np.sqrt(total / counter[label])
    )

CONFIG["class_weights"] = weights
print("\n[Weights] Smooth Class Balancing Arrays Configuration:")
for k, v in sorted(label2id.items()):
    print(f"  Class: {k:<25} | Array ID: {v} | Smooth Penalty Weight: {weights[v]:.4f}")

train_dataset = LegalSegDataset(train_docs)
val_dataset = LegalSegDataset(val_docs)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,                
    collate_fn=identity_collate
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    collate_fn=identity_collate
)

device = CONFIG["device"]
model = BaselineModel(CONFIG, label2id)
model.to(device)

optimizer = torch.optim.Adam([
    {"params": model.legalbert.parameters(), "lr": 2e-6},  
    {"params": model.bilstm.parameters(), "lr": 5e-5},      
    {"params": model.classifier.parameters(), "lr": 1e-4},  
    {"params": model.crf.parameters(), "lr": 1e-3}          
])

history = train(
    model,
    train_loader,
    val_loader,
    optimizer,
    CONFIG
)

print("\n🎉 Pipeline Routine Finalized Successfully.")

Current workspace directory : /kaggle/working

[Data] Fetching opennyaiorg/InRhetoricalRoles (train split)...
 -> Compiled 247 complete document sequences.

[Data] Fetching opennyaiorg/InRhetoricalRoles (dev split)...
 -> Compiled 30 complete document sequences.

[Weights] Smooth Class Balancing Arrays Configuration:
  Class: ANALYSIS                  | Array ID: 0 | Smooth Penalty Weight: 1.7532
  Class: ARG_PETITIONER            | Array ID: 1 | Smooth Penalty Weight: 4.8670
  Class: ARG_RESPONDENT            | Array ID: 2 | Smooth Penalty Weight: 6.8554
  Class: FAC                       | Array ID: 3 | Smooth Penalty Weight: 2.3102
  Class: ISSUE                     | Array ID: 4 | Smooth Penalty Weight: 8.7427
  Class: NONE                      | Array ID: 5 | Smooth Penalty Weight: 4.0468
  Class: None                      | Array ID: 6 | Smooth Penalty Weight: 10.2553
  Class: PREAMBLE                  | Array ID: 7 | Smooth Penalty Weight: 2.3599
  Class: PRE_NOT_RELIED         

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Starting Model Training Pipeline for 10 Epochs...
--------------------------------------------------

Epoch 1/10


  Training Batches:   0%|          | 0/247 [00:00<?, ?it/s]

...

-> Epoch 8 Results | Train Avg Loss: 57.0956 | Val Avg Loss: 50.5902
   ✨ New best validation checkpoint saved to 'best_model.pth'

Epoch 9/10


  Training Batches:   0%|          | 0/247 [00:00<?, ?it/s]

-> Epoch 9 Results | Train Avg Loss: 50.7352 | Val Avg Loss: 47.2346
   ✨ New best validation checkpoint saved to 'best_model.pth'

Epoch 10/10


  Training Batches:   0%|          | 0/247 [00:00<?, ?it/s]

-> Epoch 10 Results | Train Avg Loss: 48.1085 | Val Avg Loss: 46.6291
   ✨ New best validation checkpoint saved to 'best_model.pth'

🎉 Pipeline Routine Finalized Successfully.


# **TEST**

In [39]:


import os
import torch
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report
from tqdm.auto import tqdm  

try:
    from datasets import load_dataset
except ImportError:
    print("Installing missing datasets library...")
    os.system("pip install datasets")
    from datasets import load_dataset


def identity_collate(batch):
    return batch[0]


def load_opennyai_data(split_name="test"):
    print(f"\n[Data] Fetching opennyaiorg/InRhetoricalRoles ({split_name} split)...")
    
    hf_split = "test"
    hf_dataset = load_dataset("opennyaiorg/InRhetoricalRoles", split=hf_split)
    
    parsed_documents = []
    
    for idx, row in enumerate(hf_dataset):
        full_text = row["data"]["text"]
        annotations_list = row["annotations"]
        
        extracted_segments = []
        if annotations_list and len(annotations_list) > 0:
            results = annotations_list[0].get("result", [])
            for res in results:
                value = res.get("value", {})
                start_char = value.get("start")
                end_char = value.get("end")
                labels = value.get("labels", ["None"])
                label = labels[0] if labels else "None"
                
                segment_text = full_text[start_char:end_char].strip()
                if segment_text:
                    extracted_segments.append((segment_text, label))
                    
        doc_sentences = []
        doc_labels = []
        
        if extracted_segments:
            for seg_text, label in extracted_segments:
                import re
                sub_sentences = re.split(r'(?<=[.!?])\s+', seg_text)
                for sentence in sub_sentences:
                    sentence = sentence.strip()
                    if len(sentence) > 3:  
                        doc_sentences.append(sentence)
                        doc_labels.append(label)
        else:
            import re
            sub_sentences = re.split(r'(?<=[.!?])\s+', full_text)
            for sentence in sub_sentences:
                sentence = sentence.strip()
                if len(sentence) > 3:
                    doc_sentences.append(sentence)
                    doc_labels.append("None")
                    
        if doc_sentences:
            parsed_documents.append({
                "sentences": doc_sentences,
                "labels": doc_labels,
                "doc_id": f"opennyai_{split_name}_{idx}"
            })
            
    print(f" -> Compiled {len(parsed_documents)} complete document sequences.")
    return parsed_documents


test_docs = load_opennyai_data(split_name="test")
label2id = {
    'ANALYSIS': 0, 'ARG_PETITIONER': 1, 'ARG_RESPONDENT': 2, 'FAC': 3, 
    'ISSUE': 4, 'NONE': 5, 'PREAMBLE': 6, 'PRE_NOT_RELIED': 7, 
    'PRE_RELIED': 8, 'RATIO': 9, 'RLC': 10, 'RPC': 11, 'STA': 12, 'None': 13
}

id2label = {v: k for k, v in label2id.items()}
CONFIG["num_labels"] = len(label2id)  

test_dataset = LegalSegDataset(test_docs)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=identity_collate
)

device = CONFIG["device"]

model = BaselineModel(CONFIG, label2id)

model = load_model(model, "best_model.pth", device)

print("Using checkpoint timestamped at :", os.getcwd())

model.to(device)
model.eval()

true_labels = []
pred_labels = []

progress_bar = tqdm(
    test_loader, 
    desc="Evaluating Test Set Split", 
    dynamic_ncols=True
)

with torch.no_grad():
    for batch in progress_bar:

        output = model(batch, device)
        labels = [
            "None"
            if str(x).lower() == "nan"
            else str(x).strip()
            for x in batch["labels"]
        ]
        predictions = output["predictions"][0]

        predicted_names = [
            id2label[x]
            for x in predictions
        ]
        if len(labels) != len(predicted_names):
            if len(predicted_names) < len(labels):
                shortfall = len(labels) - len(predicted_names)
                predicted_names.extend([predicted_names[-1] if predicted_names else "NONE"] * shortfall)
            else:
                predicted_names = predicted_names[:len(labels)]

        true_labels.extend(labels)
        pred_labels.extend(predicted_names)

metrics = evaluate(true_labels, pred_labels)

print("\n" + "="*40)
print(" FINAL BENCHMARK PERFORMANCE REPORTS")
print("="*40)
print(f"Macro F1      : {metrics['macro_f1']:.4f}")
print(f"Weighted F1   : {metrics['weighted_f1']:.4f}")
print(f"Precision     : {metrics['precision']:.4f}")
print(f"Recall        : {metrics['recall']:.4f}")

print("\n--- Classification Report ---")
print(
    classification_report(
        true_labels,
        pred_labels,
        zero_division=0
    )
)


[Data] Fetching opennyaiorg/InRhetoricalRoles (test split)...
 -> Compiled 50 complete document sequences.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using checkpoint timestamped at : /kaggle/working


Evaluating Test Set Split:   0%|          | 0/50 [00:00<?, ?it/s]


 FINAL BENCHMARK PERFORMANCE REPORTS
Macro F1      : 0.3058
Weighted F1   : 0.4826
Precision     : 0.2990
Recall        : 0.3222

--- Classification Report ---
                precision    recall  f1-score   support

      ANALYSIS       0.77      0.73      0.75      1516
ARG_PETITIONER       0.45      0.63      0.52       324
ARG_RESPONDENT       0.33      0.59      0.43        95
           FAC       0.80      0.95      0.86      1261
         ISSUE       0.94      0.87      0.90        67
          NONE       0.90      0.74      0.81       399
          None       0.00      0.00      0.00         0
      PREAMBLE       0.00      0.00      0.00      1290
PRE_NOT_RELIED       0.00      0.00      0.00         2
    PRE_RELIED       0.00      0.00      0.00       263
         RATIO       0.01      0.01      0.01       153
           RLC       0.00      0.00      0.00       143
           RPC       0.00      0.00      0.00       240
           STA       0.00      0.00      0.00        9